In [1]:
import os

# ==========================================
# 0. ⚙️ GLOBAL CONFIGURATION
# ==========================================
# Change these values to control the entire script
TARGET_ROWS = 1500           # Total rows to generate (e.g., 200 or 1300000)
NUM_CORES = 19              # Number of CPU cores to use
DUPLICATE_RATIO = 0.45       # 30% of data will be duplicates
OUTPUT_FILE = 'dedupe_churn_results.csv'
SETTINGS_FILE = 'dedupe_churn_settings.settings'
TRAINING_JSON = 'dedupe_churn_training.json'

# ⚠️ WINDOWS FIX: Must be set using the config above BEFORE importing dedupe
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES)

# ==========================================
# IMPORTS (Must be after os.environ setup)
# ==========================================
import random
import datetime
import csv
import time
import multiprocessing
import json
import logging 
import threading 
import itertools 
import sys       
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

def spinner_task(stop_event):
    """Runs a visual spinner in the console to show activity."""
    spinner = itertools.cycle(['-', '/', '|', '\\'])
    while not stop_event.is_set():
        sys.stdout.write(next(spinner))
        sys.stdout.flush()
        sys.stdout.write('\b')
        time.sleep(0.1)

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows, duplicate_ratio):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    """
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 100000 == 0 and i > 0: print(f"       ...{i} base records created")

    # --- B. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Change
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            
        # Scenario 3: Missing Data
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)
        
        if i % 50000 == 0 and i > 0: print(f"       ...{i} duplicates created")

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(500): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(500):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    # Enable Detailed Logging
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    # Force Fresh Start
    if os.path.exists(TRAINING_JSON):
        print(f"🗑️ Deleting old training file: {TRAINING_JSON} (Fresh Start)")
        os.remove(TRAINING_JSON)

    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    print(f"⚡ Parallel Mode: {NUM_CORES} Cores")

    print(f"🌪️ Generating Data ({TARGET_ROWS} Rows)...")
    t_gen = time.time()
    
    # ⚙️ USING GLOBAL VARIABLES HERE
    data_d, training_data = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO) 
    
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    if os.path.exists(TRAINING_JSON):
        print(f"   Reading labeled examples from {TRAINING_JSON}...")
        with open(TRAINING_JSON, 'r') as f:
            deduper.prepare_training(data_d, training_file=f)
    else:
        print("   Using auto-generated training data...")
        deduper.prepare_training(data_d)
        deduper.mark_pairs(training_data)

    print("🎓 Starting active labeling...")
    print("   [Instructions] y: Yes, n: No, u: Unsure, f: Finished")
    try:
        dedupe.console_label(deduper)
    except dedupe.predicates.NoIndexError:
        pass 

    print(f"💾 Saving manual training to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as tf:
        deduper.write_training(tf)

    # --- 🌀 SPINNER IMPLEMENTATION START ---
    print("🎓 Training Model...", end=" ") 
    
    stop_spinner = threading.Event()
    spinner_thread = threading.Thread(target=spinner_task, args=(stop_spinner,))
    spinner_thread.start()
    
    try:
        deduper.train()
    finally:
        stop_spinner.set()
        spinner_thread.join()
    # --- 🌀 SPINNER IMPLEMENTATION END ---

    print(f"\n✅ Trained in {time.time()-t0:.2f}s")
    
    with open(SETTINGS_FILE, 'wb') as sf:
        deduper.write_settings(sf)

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🗑️ Deleting old training file: dedupe_churn_training.json (Fresh Start)
⚡ Parallel Mode: 19 Cores
🌪️ Generating Data (1500 Rows)...
   [Helper] Generating 1034 unique records...
   [Helper] Generating 466 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 0.02s
🧠 Initializing Dedupe...
   Using auto-generated training data...


Final predicate set:
SimplePredicate: (wholeFieldPredicate, name_only)
Final predicate set:
TfidfTextCanopyPredicate: (0.6, bank_acct_no)
SimplePredicate: (fingerprint, address)


🎓 Starting active labeling...
   [Instructions] y: Yes, n: No, u: Unsure, f: Finished


name_only : Kerry Group Group
gender : None
address : 654 Henry St, Bray
dob : None
occupation : None
bank_acct_no : IE53BOFI910287

name_only : Kerry Group Solutions
gender : None
address : 121 High St, Swords
dob : None
occupation : None
bank_acct_no : IE20BOFI953359

500/10 positive, 500/10 negative
Do these records refer to the same thing?
(y)es / (n)o / (u)nsure / (f)inished


 y


...

💾 Saving manual training to dedupe_churn_training.json...
🎓 Training Model... /

Finished labeling


|

Final predicate set:
LevenshteinCanopyPredicate: (4, name_only)
(SimplePredicate: (twoGramFingerprint, bank_acct_no), ExistsPredicate: (Exists, dob))
(SimplePredicate: (oneGramFingerprint, name_only), SimplePredicate: (nearIntegersPredicate, address))
SimplePredicate: (firstTwoTokensPredicate, name_only)



✅ Trained in 67.37s
🧩 Clustering...
✅ Clustered in 6.68s
💾 Saving CSV...
🎉 Done. Saved to dedupe_churn_results.csv


In [2]:
#Full pipeline with manual training/active labelling

In [3]:
import os
import random
import datetime
import csv
import time
import multiprocessing
import json
import logging 
import threading 
import itertools 
import sys
import re
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. ⚙️ CONFIGURATION
# ==========================================
TARGET_ROWS = 1600           # Keep small for testing
NUM_CORES = 19              
DUPLICATE_RATIO = 0.5446       
OUTPUT_FILE = 'final_churn_analysis.csv'
SETTINGS_FILE = 'dedupe_churn_settings.settings'
TRAINING_JSON = 'dedupe_churn_training.json'

# ⚠️ WINDOWS FIX
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES)

import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def random_policy_dates():
    """Generates policy start/end dates for Churn calculation."""
    # Policy starts sometime in 2020-2022
    start_date = random_date(2020, 2022)
    # Policy lasts 1 year
    end_date = start_date + datetime.timedelta(days=365)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

def corrupt_string(s):
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

def spinner_task(stop_event):
    spinner = itertools.cycle(['-', '/', '|', '\\'])
    while not stop_event.is_set():
        sys.stdout.write(next(spinner))
        sys.stdout.flush()
        sys.stdout.write('\b')
        time.sleep(0.1)

# ==========================================
# 2. Main Data Generator (Modified for Churn)
# ==========================================
def generate_huge_dataset(target_rows, duplicate_ratio):
    
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Phase 1] Generating {num_base_records} unique base records...")

    # --- Generate Records ---
    for i in range(num_base_records):
        counter += 1
        
        is_corporate = random.random() < 0.10
        
        # Policy Dates (Crucial for Churn Calc)
        p_start, p_end = random_policy_dates()

        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
            company_ind = 'C'
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)
            company_ind = None

        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'policy_no': f"P{counter}", # Unique Policy ID
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank,
            'company_ind': company_ind,
            'inception_date': p_start,
            'termination_date': p_end
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]

    # --- Generate Duplicates (Simulating Renewals/Churners) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Phase 1] Generating {num_dupes} duplicates (Potential Renewals)...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # New Policy ID for the duplicate (It's a new policy for same person)
        new_rec['policy_no'] = f"P{counter}"
        
        # Time Travel: New policy starts roughly when old one ends (Renewal Scenario)
        old_end = datetime.datetime.strptime(original['termination_date'], "%Y-%m-%d")
        
        # 50% chance of perfect renewal (within 0 days), 50% chance of gap (churn risk)
        gap = random.randint(-5, 30) 
        new_start = old_end + datetime.timedelta(days=gap)
        new_end = new_start + datetime.timedelta(days=365)
        
        new_rec['inception_date'] = new_start.strftime("%Y-%m-%d")
        new_rec['termination_date'] = new_end.strftime("%Y-%m-%d")

        # Noise Injection
        if random.random() > 0.6: new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- Training Data ---
    print("   [Phase 1] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(min(500, len(multi_grps))): 
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    all_ids = list(data_d.keys())
    for _ in range(500):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    # --- A. SETUP ---
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    if os.path.exists(TRAINING_JSON):
        os.remove(TRAINING_JSON)

    # Fields match the provided python script, but mapped to SQL logic later
    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    # --- B. DATA GENERATION ---
    print(f"⚡ Phase 1: Data Generation ({TARGET_ROWS} Rows)...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO) 
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # --- C. DEDUPE (CLUSTERING) ---
    print("🧠 Phase 2: Dedupe (Probabilistic Matching)...")
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    deduper.prepare_training(data_d)
    
    # Seed with ground truth to provide a baseline
    print("   Seeding with generated ground truth (fast start)...")
    deduper.mark_pairs(training_data)
    
    # --- MANUAL ACTIVE LABELING LOOP ---
    print("🎓 Starting manual active labeling loop...")
    print("   [Instructions] y: Yes, n: No, u: Unsure, f: Finished")
    print("   (Press 'f' when you are satisfied with the training examples)")
    try:
        dedupe.console_label(deduper)
    except dedupe.predicates.NoIndexError:
        print("   No more uncertain pairs to label.")
    
    # --- SAVE TRAINING (New Step) ---
    print(f"💾 Saving training data to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as tf:
        deduper.write_training(tf)
    
    print("🎓 Training Model...", end=" ") 
    stop_spinner = threading.Event()
    spinner_thread = threading.Thread(target=spinner_task, args=(stop_spinner,))
    spinner_thread.start()
    try:
        deduper.train()
    finally:
        stop_spinner.set()
        spinner_thread.join()

    # --- SAVE SETTINGS (New Step) ---
    print(f"\n💾 Saving model settings/blocking rules to {SETTINGS_FILE}...")
    with open(SETTINGS_FILE, 'wb') as sf:
        deduper.write_settings(sf)

    print("🧩 Clustering...")
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    
    # Convert Dedupe Output to DataFrame for Rule Processing
    # This mimics loading the "CUST_MTCH_TEMP_CHURN" table
    cluster_map = {}
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            cluster_map[member_id] = {'cluster_id': cluster_id, 'score': score}
            
    # Create DataFrame from data_d
    df = pd.DataFrame.from_dict(data_d, orient='index')
    df['id'] = df.index
    
    # Map clusters back to DF
    df['cluster_id'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('cluster_id', -1))
    df['score'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('score', 0))
    
    # If Dedupe didn't cluster it (singleton), give it a unique cluster ID (negative)
    df.loc[df['cluster_id'] == -1, 'cluster_id'] = df.loc[df['cluster_id'] == -1, 'id'] * -1

    # --- D. APPLY SQL BUSINESS RULES (From 'Churn all scripts.txt') ---
    print("⚖️  Phase 3: Applying SQL Business Rules (Post-Processing)...")
    
    # Helper for NULL handling (Pandas uses NaN/None, SQL uses NULL)
    df['name_short'] = df['name_only'].str.slice(0, 5)
    df['addr_short'] = df['address'].str.slice(0, 10)
    df['occ_short'] = df['occupation'].str.slice(0, 10)
    df['dob_filled'] = df['dob'].fillna('None')
    df['bank_filled'] = df['bank_acct_no'].fillna('None')
    
    # Rule 0: Confidence Score Threshold (From SQL: CASE WHEN SCORE >= 0.7 ...)
    # If the ML model isn't 70% sure, we treat them as unique (reset to original ID)
    # Note: We multiply by -1 to ensure they don't accidentally match existing positive cluster IDs
    mask_low_conf = (df['score'] < 0.7) & (df['score'] > 0.0) 
    if mask_low_conf.any():
        print(f"   [Rule 0] Resetting {mask_low_conf.sum()} matches with score < 0.7 (SQL Rule)")
        df.loc[mask_low_conf, 'cluster_id'] = df.loc[mask_low_conf, 'id'] * -1

    # 1. Rule: Merge on Name + Bank + DOB (where Bank exists)
    # SQL: MIN(CLUSTER_ID) OVER(PARTITION BY NAME_ONLY, DOB, BANK_ACCT_NO)
    mask = df['bank_acct_no'].notna()
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'dob_filled', 'bank_acct_no'])['cluster_id'].transform('min')

    # 2. Rule: Merge on Substring Address + Substring Name + DOB
    # SQL: MIN(CLUSTER_ID) OVER(PARTITION BY SUBSTR(ADDRESS,1,10), SUBSTR(NAME,1,5), DOB)
    mask = (df['address'].notna()) & (df['name_only'].notna())
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['addr_short', 'name_short', 'dob_filled'])['cluster_id'].transform('min')

    # 3. Rule: Corporate Merge (Name only)
    # SQL: WHERE COMPANY_IND='C' ... PARTITION BY NAME_ONLY
    mask = df['company_ind'] == 'C'
    if mask.any():
        df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only'])['cluster_id'].transform('min')

    # 4. Rule: Occupation Merge
    # SQL: PARTITION BY NAME_ONLY, DOB, SUBSTR(OCCUPATION,1,10)
    df['cluster_id'] = df.groupby(['name_only', 'dob_filled', 'occ_short'])['cluster_id'].transform('min')

    # 5. Rule: Complex Date Logic (Clients with same Name/Address but DOBs within 10 years)
    def merge_fuzzy_dates(group):
        # Extract years from valid DOBs
        valid_dobs = pd.to_datetime(group['dob'], errors='coerce').dropna()
        if len(valid_dobs) > 1:
            min_year = valid_dobs.min().year
            max_year = valid_dobs.max().year
            if (max_year - min_year) <= 10:
                return group['cluster_id'].min() # Merge them
        return group['cluster_id'] # Keep as is

    # Apply complex date rule group by Name + Address
    # (Only doing this for non-null addresses/names)
    mask = (df['address'].notna()) & (df['name_only'].notna())
    # Note: This operation can be slow on millions of rows, but fine for 200
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'address'], group_keys=False).apply(lambda x: x.assign(cluster_id=merge_fuzzy_dates(x)))['cluster_id']

    # --- E. CHURN CALCULATION (From 'Churn all scripts.txt') ---
    print("📉 Phase 4: Calculating Churn...")

    # We treat the DataFrame as both the Customer Map and the Policy Data
    # Self-Join on Cluster ID to compare policies held by the "Same Person"
    
    churn_df = pd.merge(
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        on='cluster_id',
        suffixes=('_old', '_new')
    )

    # Filter: We want to compare distinct policies
    churn_df = churn_df[churn_df['policy_no_old'] != churn_df['policy_no_new']]

    # Logic: diff = New Inception - Old Termination
    # SQL: coalesce(b.termination_date, '2099-01-01')
    churn_df['term_date_filled'] = pd.to_datetime(churn_df['termination_date_old']).fillna(pd.Timestamp('2099-01-01'))
    churn_df['incept_date_new'] = pd.to_datetime(churn_df['inception_date_new'])
    
    churn_df['diff_days'] = (churn_df['incept_date_new'] - churn_df['term_date_filled']).dt.days

    # SQL Rule: where diff < 3 and diff > -21 (This defines a RENEWAL)
    # If they fall in this window, they Renewed. If not, they Churned.
    renewals = churn_df[
        (churn_df['diff_days'] < 3) & 
        (churn_df['diff_days'] > -21)
    ]
    
    # Identify Churners: Policies that expired but have NO matching renewal in the window
    # 1. Get all expired policies
    today = pd.Timestamp('2025-01-01') # Simulation run date
    expired_policies = df[pd.to_datetime(df['termination_date']) < today]['policy_no']
    
    # 2. Get policies that were successfully renewed
    renewed_policy_ids = renewals['policy_no_old'].unique()
    
    # 3. Churners = Expired - Renewed
    churned_policy_ids = set(expired_policies) - set(renewed_policy_ids)

    # Mark rows in main DF
    df['Status'] = 'Active'
    df.loc[df['policy_no'].isin(churned_policy_ids), 'Status'] = 'CHURNED'
    df.loc[df['policy_no'].isin(renewed_policy_ids), 'Status'] = 'Renewed'

    # --- F. SAVING RESULTS ---
    print(f"💾 Saving Analysis to {OUTPUT_FILE}...")
    
    output_cols = ['cluster_id', 'Status', 'policy_no', 'name_only', 'address', 'dob', 'bank_acct_no', 'inception_date', 'termination_date']
    df[output_cols].sort_values(by=['cluster_id', 'inception_date']).to_csv(OUTPUT_FILE, index=False)

    # Stats
    total_churners = df[df['Status'] == 'CHURNED'].shape[0]
    total_renewals = df[df['Status'] == 'Renewed'].shape[0]
    
    print("-" * 30)
    print(f"📊 REPORT SUMMARY")
    print("-" * 30)
    print(f"Total Policies:   {len(df)}")
    print(f"Unique Clusters:  {df['cluster_id'].nunique()}")
    print(f"Identified Churn: {total_churners}")
    print(f"Identified Renew: {total_renewals}")
    print(f"Active/Other:     {len(df) - total_churners - total_renewals}")
    print("-" * 30)
    print(f"🎉 Pipeline Complete.")

⚡ Phase 1: Data Generation (1600 Rows)...
   [Phase 1] Generating 1035 unique base records...
   [Phase 1] Generating 565 duplicates (Potential Renewals)...
   [Phase 1] Generating training pairs...
✅ Generated in 0.04s
🧠 Phase 2: Dedupe (Probabilistic Matching)...


Final predicate set:
Final predicate set:
SimplePredicate: (wholeFieldPredicate, name_only)
SimplePredicate: (wholeFieldPredicate, name_only)


   Seeding with generated ground truth (fast start)...


Final predicate set:
Final predicate set:
TfidfTextCanopyPredicate: (0.6, bank_acct_no)
TfidfTextCanopyPredicate: (0.6, bank_acct_no)
SimplePredicate: (fingerprint, address)
SimplePredicate: (fingerprint, address)


🎓 Starting manual active labeling loop...
   [Instructions] y: Yes, n: No, u: Unsure, f: Finished
   (Press 'f' when you are satisfied with the training examples)


...

 f


Finished labeling
Final predicate set:
Final predicate set:
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
SimplePredicate: (fingerprint, address)
SimplePredicate: (fingerprint, address)
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)


💾 Saving training data to dedupe_churn_training.json...
🎓 Training Model... 

Final predicate set:
Final predicate set:
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)



💾 Saving model settings/blocking rules to dedupe_churn_settings.settings...
🧩 Clustering...
⚖️  Phase 3: Applying SQL Business Rules (Post-Processing)...
   [Rule 0] Resetting 145 matches with score < 0.7 (SQL Rule)
📉 Phase 4: Calculating Churn...
💾 Saving Analysis to final_churn_analysis.csv...
------------------------------
📊 REPORT SUMMARY
------------------------------
Total Policies:   1600
Unique Clusters:  931
Identified Churn: 1322
Identified Renew: 274
Active/Other:     4
------------------------------
🎉 Pipeline Complete.


In [4]:
#Loading the trained settings

In [5]:
import os
import random
import datetime
import csv
import time
import multiprocessing
import json
import logging 
import threading 
import itertools 
import sys
import re
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')


# ==========================================
# 0. ⚙️ PRODUCTION CONFIGURATION
# ==========================================
TARGET_ROWS = 1300       # 🚀 PRODUCTION SCALE
NUM_CORES = 19              
DUPLICATE_RATIO = 0.4524       
OUTPUT_FILE = 'production_results_1.3M.csv'
SETTINGS_FILE = 'dedupe_churn_settings.settings' # Must exist from previous run!

# ⚠️ WINDOWS FIX
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES)

import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities (Same as Pipeline)
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def random_policy_dates():
    """Generates policy start/end dates for Churn calculation."""
    start_date = random_date(2020, 2022)
    end_date = start_date + datetime.timedelta(days=365)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

def corrupt_string(s):
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

def spinner_task(stop_event):
    spinner = itertools.cycle(['-', '/', '|', '\\'])
    while not stop_event.is_set():
        sys.stdout.write(next(spinner))
        sys.stdout.flush()
        sys.stdout.write('\b')
        time.sleep(0.1)

# ==========================================
# 2. Production Data Generator
# ==========================================
def generate_huge_dataset(target_rows, duplicate_ratio):
    """
    Generates 1.3 Million rows efficiently.
    Does NOT generate training pairs (not needed for production).
    """
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Phase 1] Generating {num_base_records:,} unique base records...")

    # --- Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        is_corporate = random.random() < 0.10
        p_start, p_end = random_policy_dates()

        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
            company_ind = 'C'
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)
            company_ind = None

        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'policy_no': f"P{counter}",
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank,
            'company_ind': company_ind,
            'inception_date': p_start,
            'termination_date': p_end
        }
        data_d[counter] = record
        
        if i % 100000 == 0 and i > 0: print(f"       ...{i:,} records created")

    # --- Generate Duplicates (Renewals/Churners) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Phase 1] Generating {num_dupes:,} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        new_rec['policy_no'] = f"P{counter}"
        
        # Time Travel Logic
        old_end = datetime.datetime.strptime(original['termination_date'], "%Y-%m-%d")
        gap = random.randint(-5, 30) 
        new_start = old_end + datetime.timedelta(days=gap)
        new_end = new_start + datetime.timedelta(days=365)
        
        new_rec['inception_date'] = new_start.strftime("%Y-%m-%d")
        new_rec['termination_date'] = new_end.strftime("%Y-%m-%d")

        # Noise Injection
        if random.random() > 0.6: new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None

        data_d[counter] = new_rec
        
        if i % 50000 == 0 and i > 0: print(f"       ...{i:,} duplicates created")

    return data_d

# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    # 🛑 CHECK FOR SETTINGS FILE
    if not os.path.exists(SETTINGS_FILE):
        print("❌ ERROR: Settings file not found!")
        print(f"   Please run 'End_to_End_Churn_Pipeline.py' first to train the model and generate '{SETTINGS_FILE}'.")
        sys.exit(1)

    print(f"⚡ Phase 1: Generating {TARGET_ROWS:,} Rows for Production...")
    t_gen = time.time()
    data_d = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO) 
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # --- LOAD STATIC DEDUPE (No Training) ---
    print(f"🧠 Phase 2: Loading Trained Model from {SETTINGS_FILE}...")
    t0 = time.time()
    
    with open(SETTINGS_FILE, 'rb') as f:
        deduper = dedupe.StaticDedupe(f, num_cores=NUM_CORES)

    print("🧩 Clustering (This may take a while)...", end=" ")
    
    stop_spinner = threading.Event()
    spinner_thread = threading.Thread(target=spinner_task, args=(stop_spinner,))
    spinner_thread.start()
    
    try:
        # Threshold hardcoded or previously learned. Keeping 0.5 as per previous request/logic.
        clustered_dupes = deduper.partition(data_d, threshold=0.5)
    finally:
        stop_spinner.set()
        spinner_thread.join()
        
    print(f"\n✅ Clustered in {time.time()-t0:.2f}s")
    
    # Convert to DataFrame
    print("   Mapping clusters to DataFrame...")
    cluster_map = {}
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            cluster_map[member_id] = {'cluster_id': cluster_id, 'score': score}
            
    df = pd.DataFrame.from_dict(data_d, orient='index')
    df['id'] = df.index
    df['cluster_id'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('cluster_id', -1))
    df['score'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('score', 0))
    df.loc[df['cluster_id'] == -1, 'cluster_id'] = df.loc[df['cluster_id'] == -1, 'id'] * -1

    # --- APPLY SQL RULES ---
    print("⚖️  Phase 3: Applying SQL Business Rules...")
    
    df['name_short'] = df['name_only'].str.slice(0, 5)
    df['addr_short'] = df['address'].str.slice(0, 10)
    df['occ_short'] = df['occupation'].str.slice(0, 10)
    df['dob_filled'] = df['dob'].fillna('None')
    
    # Rule 0: Confidence < 0.7
    mask_low_conf = (df['score'] < 0.7) & (df['score'] > 0.0) 
    if mask_low_conf.any():
        df.loc[mask_low_conf, 'cluster_id'] = df.loc[mask_low_conf, 'id'] * -1

    # Rule 1: Name + Bank + DOB
    mask = df['bank_acct_no'].notna()
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'dob_filled', 'bank_acct_no'])['cluster_id'].transform('min')

    # Rule 2: Address + Name + DOB
    mask = (df['address'].notna()) & (df['name_only'].notna())
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['addr_short', 'name_short', 'dob_filled'])['cluster_id'].transform('min')

    # Rule 3: Corporate
    mask = df['company_ind'] == 'C'
    if mask.any():
        df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only'])['cluster_id'].transform('min')

    # Rule 4: Occupation
    df['cluster_id'] = df.groupby(['name_only', 'dob_filled', 'occ_short'])['cluster_id'].transform('min')

    # Rule 5: Complex Date Logic (Slowest part)
    print("   Applying fuzzy date logic (may be slow)...")
    def merge_fuzzy_dates(group):
        valid_dobs = pd.to_datetime(group['dob'], errors='coerce').dropna()
        if len(valid_dobs) > 1:
            if (valid_dobs.max().year - valid_dobs.min().year) <= 10:
                return group['cluster_id'].min()
        return group['cluster_id']

    mask = (df['address'].notna()) & (df['name_only'].notna())
    # Optimize: Only apply to groups > 1 size to save time
    # (Simple apply here for consistency with previous script)
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'address'], group_keys=False).apply(lambda x: x.assign(cluster_id=merge_fuzzy_dates(x)))['cluster_id']

    # --- CHURN CALCULATION ---
    print("📉 Phase 4: Calculating Churn...")
    
    churn_df = pd.merge(
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        on='cluster_id',
        suffixes=('_old', '_new')
    )
    churn_df = churn_df[churn_df['policy_no_old'] != churn_df['policy_no_new']]
    
    churn_df['term_date_filled'] = pd.to_datetime(churn_df['termination_date_old']).fillna(pd.Timestamp('2099-01-01'))
    churn_df['incept_date_new'] = pd.to_datetime(churn_df['inception_date_new'])
    churn_df['diff_days'] = (churn_df['incept_date_new'] - churn_df['term_date_filled']).dt.days

    renewals = churn_df[(churn_df['diff_days'] < 3) & (churn_df['diff_days'] > -21)]
    
    today = pd.Timestamp('2025-01-01')
    expired_policies = df[pd.to_datetime(df['termination_date']) < today]['policy_no']
    renewed_policy_ids = renewals['policy_no_old'].unique()
    churned_policy_ids = set(expired_policies) - set(renewed_policy_ids)

    df['Status'] = 'Active'
    df.loc[df['policy_no'].isin(churned_policy_ids), 'Status'] = 'CHURNED'
    df.loc[df['policy_no'].isin(renewed_policy_ids), 'Status'] = 'Renewed'

    # --- SAVE ---
    print(f"💾 Saving 1.3M records to {OUTPUT_FILE}...")
    output_cols = ['cluster_id', 'Status', 'policy_no', 'name_only', 'address', 'dob', 'bank_acct_no', 'inception_date', 'termination_date']
    
    # Save in chunks to be safe with memory
    df[output_cols].sort_values(by=['cluster_id']).to_csv(OUTPUT_FILE, index=False)

    print("-" * 30)
    print(f"📊 PRODUCTION SUMMARY (1.3M Records)")
    print("-" * 30)
    print(f"Total Churners: {df[df['Status'] == 'CHURNED'].shape[0]}")
    print(f"Total Renewals: {df[df['Status'] == 'Renewed'].shape[0]}")
    print("-" * 30)
    print("Done.")

Predicate set:
Predicate set:
Predicate set:
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)


⚡ Phase 1: Generating 1,300 Rows for Production...
   [Phase 1] Generating 895 unique base records...
   [Phase 1] Generating 405 duplicates...
✅ Generated in 0.03s
🧠 Phase 2: Loading Trained Model from dedupe_churn_settings.settings...
🧩 Clustering (This may take a while)... 
✅ Clustered in 9.30s
   Mapping clusters to DataFrame...
⚖️  Phase 3: Applying SQL Business Rules...
   Applying fuzzy date logic (may be slow)...
📉 Phase 4: Calculating Churn...
💾 Saving 1.3M records to production_results_1.3M.csv...
------------------------------
📊 PRODUCTION SUMMARY (1.3M Records)
------------------------------
Total Churners: 1084
Total Renewals: 214
------------------------------
Done.


In [6]:
#V2 all business logic implemented

In [7]:
import os
import random
import datetime
import csv
import time
import multiprocessing
import json
import logging 
import threading 
import itertools 
import sys
import re
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')


# ==========================================
# 0. ⚙️ PRODUCTION CONFIGURATION
# ==========================================
TARGET_ROWS = 5000       # 🚀 PRODUCTION SCALE
NUM_CORES = 19              
DUPLICATE_RATIO = 0.4       
OUTPUT_FILE = 'production_results_1.3M.csv'
SETTINGS_FILE = 'dedupe_churn_settings.settings' # Must exist from previous run!

# ⚠️ WINDOWS FIX
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES)

import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def random_policy_dates():
    start_date = random_date(2020, 2022)
    end_date = start_date + datetime.timedelta(days=365)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

def corrupt_string(s):
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

def spinner_task(stop_event):
    spinner = itertools.cycle(['-', '/', '|', '\\'])
    while not stop_event.is_set():
        sys.stdout.write(next(spinner))
        sys.stdout.flush()
        sys.stdout.write('\b')
        time.sleep(0.1)

# ==========================================
# 2. Production Data Generator
# ==========================================
def generate_huge_dataset(target_rows, duplicate_ratio):
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Phase 1] Generating {num_base_records:,} unique base records...")

    # --- Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        is_corporate = random.random() < 0.10
        p_start, p_end = random_policy_dates()

        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
            company_ind = 'C'
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)
            company_ind = None

        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'policy_no': f"P{counter}",
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank,
            'company_ind': company_ind,
            'inception_date': p_start,
            'termination_date': p_end
        }
        data_d[counter] = record
        if i % 100000 == 0 and i > 0: print(f"       ...{i:,} records created")

    # --- Generate Duplicates (Renewals/Churners) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Phase 1] Generating {num_dupes:,} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        new_rec['policy_no'] = f"P{counter}"
        
        old_end = datetime.datetime.strptime(original['termination_date'], "%Y-%m-%d")
        gap = random.randint(-5, 30) 
        new_start = old_end + datetime.timedelta(days=gap)
        new_end = new_start + datetime.timedelta(days=365)
        
        new_rec['inception_date'] = new_start.strftime("%Y-%m-%d")
        new_rec['termination_date'] = new_end.strftime("%Y-%m-%d")

        if random.random() > 0.6: new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None

        data_d[counter] = new_rec
        if i % 50000 == 0 and i > 0: print(f"       ...{i:,} duplicates created")

    return data_d

# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    # 🛑 CHECK FOR SETTINGS FILE
    if not os.path.exists(SETTINGS_FILE):
        print("❌ ERROR: Settings file not found!")
        print(f"   Please run 'End_to_End_Churn_Pipeline.py' first to train the model.")
        sys.exit(1)

    print(f"⚡ Phase 1: Generating {TARGET_ROWS:,} Rows for Production...")
    t_gen = time.time()
    data_d = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO) 
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # --- LOAD STATIC DEDUPE ---
    print(f"🧠 Phase 2: Loading Trained Model from {SETTINGS_FILE}...")
    t0 = time.time()
    with open(SETTINGS_FILE, 'rb') as f:
        deduper = dedupe.StaticDedupe(f, num_cores=NUM_CORES)

    print("🧩 Clustering (This may take a while)...", end=" ")
    stop_spinner = threading.Event()
    spinner_thread = threading.Thread(target=spinner_task, args=(stop_spinner,))
    spinner_thread.start()
    
    try:
        clustered_dupes = deduper.partition(data_d, threshold=0.5)
    finally:
        stop_spinner.set()
        spinner_thread.join()
        
    print(f"\n✅ Clustered in {time.time()-t0:.2f}s")
    
    # Convert to DataFrame
    print("   Mapping clusters to DataFrame...")
    cluster_map = {}
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            cluster_map[member_id] = {'cluster_id': cluster_id, 'score': score}
            
    df = pd.DataFrame.from_dict(data_d, orient='index')
    df['id'] = df.index
    df['cluster_id'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('cluster_id', -1))
    df['score'] = df['id'].map(lambda x: cluster_map.get(x, {}).get('score', 0))
    df.loc[df['cluster_id'] == -1, 'cluster_id'] = df.loc[df['cluster_id'] == -1, 'id'] * -1

    # =========================================================================
    # ⚖️ Phase 3: SQL BUSINESS RULES (EXACT MATCH)
    # =========================================================================
    print("⚖️  Phase 3: Applying Exact SQL Business Rules...")
    
    df['name_short'] = df['name_only'].str.slice(0, 5)
    df['addr_short'] = df['address'].str.slice(0, 10)
    df['occ_short'] = df['occupation'].str.slice(0, 10)
    df['dob_filled'] = df['dob'].fillna('None')
    
    # RULE 0: Confidence Score (SQL: CASE WHEN E.CLUSTER_SCORE>=0.7 ...)
    mask_low_conf = (df['score'] < 0.7) & (df['score'] > 0.0) 
    if mask_low_conf.any():
        df.loc[mask_low_conf, 'cluster_id'] = df.loc[mask_low_conf, 'id'] * -1

    # RULE 1: Name + Bank + DOB
    # SQL: PARTITION BY NAME_ONLY, COALESCE(DOB,'None'), BANK_ACCT_NO WHERE BANK IS NOT NULL
    mask = df['bank_acct_no'].notna()
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'dob_filled', 'bank_acct_no'])['cluster_id'].transform('min')

    # RULE 2: Address(10) + Name(5) + DOB
    # SQL: PARTITION BY SUBSTR(ADDRESS,1,10), SUBSTR(NAME,1,5), DOB WHERE ADDR/NAME NOT NULL
    mask = (df['address'].notna()) & (df['name_only'].notna())
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['addr_short', 'name_short', 'dob_filled'])['cluster_id'].transform('min')

    # RULE 3: Corporate (Company Ind 'C')
    # SQL: PARTITION BY NAME_ONLY WHERE COMPANY_IND='C'
    mask = df['company_ind'] == 'C'
    if mask.any():
        df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only'])['cluster_id'].transform('min')

    # RULE 4: Occupation (With Dummy Date Exclusions)
    # SQL: WHERE DOB NOT IN ('1950-01-01'...) AND DOB IS NOT NULL
    # This prevents merging distinct people who just happen to share a generic default birthday
    dummy_dates = ['1950-01-01', '1960-01-01', '1970-01-01']
    mask_valid_dob = (df['dob'].notna()) & (~df['dob'].isin(dummy_dates))
    df.loc[mask_valid_dob, 'cluster_id'] = df.loc[mask_valid_dob].groupby(['name_only', 'dob_filled', 'occ_short'])['cluster_id'].transform('min')

    # RULE 5: Occupation (Where DOB IS NULL)
    # SQL: WHERE DOB IS NULL
    mask_null_dob = df['dob'].isna()
    df.loc[mask_null_dob, 'cluster_id'] = df.loc[mask_null_dob].groupby(['name_only', 'occ_short'])['cluster_id'].transform('min')

    # RULE 6: Substring Name + Bank (Where DOB IS NULL)
    # SQL: PARTITION BY SUBSTR(NAME,1,5), BANK WHERE DOB IS NULL AND BANK IS NOT NULL
    mask_r6 = (df['dob'].isna()) & (df['bank_acct_no'].notna())
    df.loc[mask_r6, 'cluster_id'] = df.loc[mask_r6].groupby(['name_short', 'bank_acct_no'])['cluster_id'].transform('min')

    # RULE 7: Name + Address + Occupation
    # SQL: PARTITION BY NAME, ADDRESS, OCCUPATION WHERE DOB NOT IN DUMMY DATES
    mask_r7 = (~df['dob'].isin(dummy_dates)) & (df['occupation'].notna()) & (df['address'].notna())
    df.loc[mask_r7, 'cluster_id'] = df.loc[mask_r7].groupby(['name_only', 'address', 'occupation'])['cluster_id'].transform('min')

    # RULE 8: Complex Date Logic (<= 10 years difference)
    print("   Applying fuzzy date logic (Rule 8)...")
    def merge_fuzzy_dates(group):
        valid_dobs = pd.to_datetime(group['dob'], errors='coerce').dropna()
        if len(valid_dobs) > 1:
            if (valid_dobs.max().year - valid_dobs.min().year) <= 10:
                return group['cluster_id'].min()
        return group['cluster_id']

    mask = (df['address'].notna()) & (df['name_only'].notna())
    df.loc[mask, 'cluster_id'] = df.loc[mask].groupby(['name_only', 'address'], group_keys=False).apply(lambda x: x.assign(cluster_id=merge_fuzzy_dates(x)))['cluster_id']

    # --- CHURN CALCULATION ---
    print("📉 Phase 4: Calculating Churn...")
    
    churn_df = pd.merge(
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        df[['cluster_id', 'policy_no', 'inception_date', 'termination_date']],
        on='cluster_id',
        suffixes=('_old', '_new')
    )
    churn_df = churn_df[churn_df['policy_no_old'] != churn_df['policy_no_new']]
    
    churn_df['term_date_filled'] = pd.to_datetime(churn_df['termination_date_old']).fillna(pd.Timestamp('2099-01-01'))
    churn_df['incept_date_new'] = pd.to_datetime(churn_df['inception_date_new'])
    churn_df['diff_days'] = (churn_df['incept_date_new'] - churn_df['term_date_filled']).dt.days

    # SQL: where diff < 3 and diff > -21
    renewals = churn_df[(churn_df['diff_days'] < 3) & (churn_df['diff_days'] > -21)]
    
    today = pd.Timestamp('2025-01-01')
    expired_policies = df[pd.to_datetime(df['termination_date']) < today]['policy_no']
    renewed_policy_ids = renewals['policy_no_old'].unique()
    churned_policy_ids = set(expired_policies) - set(renewed_policy_ids)

    df['Status'] = 'Active'
    df.loc[df['policy_no'].isin(churned_policy_ids), 'Status'] = 'CHURNED'
    df.loc[df['policy_no'].isin(renewed_policy_ids), 'Status'] = 'Renewed'

    # --- SAVE ---
    print(f"💾 Saving 1.3M records to {OUTPUT_FILE}...")
    output_cols = ['cluster_id', 'Status', 'policy_no', 'name_only', 'address', 'dob', 'bank_acct_no', 'inception_date', 'termination_date']
    df[output_cols].sort_values(by=['cluster_id']).to_csv(OUTPUT_FILE, index=False)

    print("-" * 30)
    print(f"📊 PRODUCTION SUMMARY")
    print("-" * 30)
    print(f"Total Churners: {df[df['Status'] == 'CHURNED'].shape[0]}")
    print(f"Total Renewals: {df[df['Status'] == 'Renewed'].shape[0]}")
    print("-" * 30)
    print("Done.")

⚡ Phase 1: Generating 5,000 Rows for Production...
   [Phase 1] Generating 3,571 unique base records...
   [Phase 1] Generating 1,429 duplicates...
✅ Generated in 0.12s
🧠 Phase 2: Loading Trained Model from dedupe_churn_settings.settings...


Predicate set:
Predicate set:
Predicate set:
Predicate set:
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
(TfidfTextCanopyPredicate: (0.8, address), SimplePredicate: (oneGramFingerprint, name_only))
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
LevenshteinCanopyPredicate: (1, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)
TfidfTextCanopyPredicate: (0.4, name_only)


🧩 Clustering (This may take a while)... /

Removing stop word St
Removing stop word St
Removing stop word St
Removing stop word St



✅ Clustered in 7.80s
   Mapping clusters to DataFrame...
⚖️  Phase 3: Applying Exact SQL Business Rules...
   Applying fuzzy date logic (Rule 8)...
📉 Phase 4: Calculating Churn...
💾 Saving 1.3M records to production_results_1.3M.csv...
------------------------------
📊 PRODUCTION SUMMARY
------------------------------
Total Churners: 4156
Total Renewals: 830
------------------------------
Done.


In [ ]:
#autotraining a machine-learning model

In [9]:
# %% [markdown]
# # 🚀 Production Dedupe Run (1.3M Records) - Simulation Mode
# *Optimized for Jupyter Notebook Execution with Profiling & Auto-Training*
#
# **Usage:**
# - Run cells sequentially.
# - If `PSQL_life_gi_data__learned_settings` exists, it loads the model.
# - If **NOT**, it auto-trains using synthetic heuristics (Active Learning Simulation).
# - Monitor the "Log_files" directory for detailed logs.
# - **Note:** DB logic removed. Data is generated in-memory and results saved to CSV.

# %% [markdown]
# ### 🧱 Cell 1: Imports, Configuration & Profiling Tools
# *Sets up libraries, multi-core, logging, and the performance profiler.*

# %%
import configparser
import datetime
import logging
import os
import sys
import time
import contextlib
import tracemalloc
import pickle
import random
import csv
import io
import threading

# [FIX] Force UTF-8 encoding for stdout/stderr to prevent Windows UnicodeEncodeError
# Note: In some Jupyter environments, this might not have a 'buffer' attribute.
# We wrap it in a try-except block to be safe.
try:
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')
except AttributeError:
    pass # Jupyter's OutStream doesn't support buffer wrapping, which is fine as it handles utf-8.

# Third-party libraries
import dedupe
import dedupe.variables # [ADDED] Required for new API

# Try importing optimization libraries
try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False

try:
    from rapidfuzz import fuzz
    FUZZY_LIB = "rapidfuzz"
except ImportError:
    from difflib import SequenceMatcher
    def fuzz_ratio(a, b): return int(SequenceMatcher(None, a, b).ratio() * 100)
    def fuzz_partial_ratio(a, b): return int(SequenceMatcher(None, a, b).ratio() * 100)
    FUZZY_LIB = "difflib"
else:
    def fuzz_ratio(a, b): return int(fuzz.ratio(a, b))
    def fuzz_partial_ratio(a, b): return int(fuzz.partial_ratio(a, b))

# ==========================================
# ⚙️ Multi-core & Heuristic Configuration
# ==========================================
NUM_CORES = 18
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES) # Windows Fix

# Heuristics for Auto-Training (from your screenshots)
SAMPLE_SIZE = 2000     # Sample size for training
MATCH_MAX = 300         # Max synthetic matches to generate
DISTINCT_MAX = 600      # Max synthetic distinct pairs
NAME_MATCH_RATIO = 90
ADDR_MATCH_RATIO = 85
NAME_DISTINCT_RATIO = 40
ADDR_DISTINCT_RATIO = 35

# %%
# [MODIFIED] Configuration for Data Generation
TARGET_ROWS = 1300000  # <--- CHANGE THIS VALUE (e.g., 200, 50000, 1300000)
DUPLICATE_RATIO = 0.38797


# ==========================================
# ⚙️ File Configuration
# ==========================================
SETTINGS_FILE = 'PSQL_life_gi_data__learned_settings'
OUTPUT_FILE = 'production_results_1.3M.csv'
LOG_DIR = 'Log_files'

# ==========================================
# 📝 Logging & Profiling Setup
# ==========================================
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

logger = logging.getLogger()
if logger.hasHandlers():
    logger.handlers.clear()
logger.setLevel(logging.INFO)

# Console Handler
# Use sys.stdout directly. If wrapped above, it's UTF-8. If Jupyter OutStream, it handles UTF-8.
ch = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
ch.setFormatter(formatter)
logger.addHandler(ch)

# File Handler
log_filename = os.path.join(LOG_DIR, f"Dedupe_Run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
fh = logging.FileHandler(log_filename, encoding='utf-8')
fh.setFormatter(formatter)
logger.addHandler(fh)

logger.info(f"✅ Environment setup complete. Asking for {NUM_CORES} Cores.")
if PSUTIL_AVAILABLE:
    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    logger.info(f"💻 System Hardware: {physical_cores} Physical Cores, {logical_cores} Logical Cores available.")
logger.info(f"📚 Fuzzy Library: {FUZZY_LIB}")

# --- Performance Profiling Utility (From Screenshots) ---
def _human_bytes(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024.0:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} PB"

def _rss_bytes():
    if PSUTIL_AVAILABLE:
        process = psutil.Process(os.getpid())
        return process.memory_info().rss
    return 0

def monitor_cpu_usage(stop_event, interval=2):
    """Background thread to monitor CPU usage per core."""
    if not PSUTIL_AVAILABLE:
        return

    logger.info("🕵️ CPU Monitor Started...")
    while not stop_event.is_set():
        # Get usage per core
        cpu_per_core = psutil.cpu_percent(interval=interval, percpu=True)
        avg_cpu = sum(cpu_per_core) / len(cpu_per_core)
        
        # Format for logging
        core_usage_str = " | ".join([f"C{i}:{u}%" for i, u in enumerate(cpu_per_core)])
        
        # Only log if there's significant activity
        if avg_cpu > 10:
            logger.info(f"🔥 CPU Load: {avg_cpu:.1f}% Avg | {core_usage_str}")
        
        # Brief sleep is handled by interval in cpu_percent

@contextlib.contextmanager
def profile_step(step_name, monitor_cpu=False):
    """
    Context manager to log execution time and memory usage for a block of code.
    Optionally starts a background thread to log CPU core usage.
    """
    logger.info(f"⏱️ Starting '{step_name}'...")
    t0 = time.perf_counter()
    rss0 = _rss_bytes()
    tracemalloc.start()
    
    stop_cpu_monitor = threading.Event()
    cpu_thread = None
    
    if monitor_cpu and PSUTIL_AVAILABLE:
        cpu_thread = threading.Thread(target=monitor_cpu_usage, args=(stop_cpu_monitor,))
        cpu_thread.start()
    
    try:
        yield
    finally:
        if cpu_thread:
            stop_cpu_monitor.set()
            cpu_thread.join()
            
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        t1 = time.perf_counter()
        rss1 = _rss_bytes()
        rss_delta = rss1 - rss0
        
        logger.info(
            f"🏁 '{step_name}' finished in {t1 - t0:.2f}s | "
            f"RSS Δ: {_human_bytes(rss_delta)} | "
            f"PyMem Peak: {_human_bytes(peak)}"
        )


# %% [markdown]
# ### 🧱 Cell 2: Helper Utilities
# *Text normalization and random data generation.*

# %%
def _norm_str(s):
    """Normalize string for fuzzy comparison."""
    return str(s).lower().strip() if s else ""

def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def random_policy_dates():
    start_date = random_date(2020, 2022)
    end_date = start_date + datetime.timedelta(days=365)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

def corrupt_string(s):
    """Introduce typos/noise into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        # Swap
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        # Delete
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"


# %% [markdown]
# ### 🧱 Cell 3: Generate Synthetic Data
# *Generates 1.3M records in memory to simulate production load.*



def generate_huge_dataset(target_rows, duplicate_ratio):
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Phase 1] Generating {num_base_records:,} unique base records...")

    # --- Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        is_corporate = random.random() < 0.10
        p_start, p_end = random_policy_dates()

        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
            company_ind = 'C'
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)
            company_ind = None

        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'policy_no': f"P{counter}",
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank,
            'company_ind': company_ind,
            'inception_date': p_start,
            'termination_date': p_end
        }
        # We use str(counter) as ID to simulate DB IDs
        data_d[str(counter)] = record
        
        # Only log progress if generating a significant amount of data
        if target_rows > 10000 and i % 10000 == 0 and i > 0: 
             print(f"       ...{i:,} records created")

    # --- Generate Duplicates (Renewals/Churners) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Phase 1] Generating {num_dupes:,} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        new_rec['policy_no'] = f"P{counter}"
        
        old_end = datetime.datetime.strptime(original['termination_date'], "%Y-%m-%d")
        gap = random.randint(-5, 30) 
        new_start = old_end + datetime.timedelta(days=gap)
        new_end = new_start + datetime.timedelta(days=365)
        
        new_rec['inception_date'] = new_start.strftime("%Y-%m-%d")
        new_rec['termination_date'] = new_end.strftime("%Y-%m-%d")

        if random.random() > 0.6: new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None

        data_d[str(counter)] = new_rec
        
        if target_rows > 10000 and i % 10000 == 0 and i > 0: 
            print(f"       ...{i:,} duplicates created")

    return data_d

with profile_step("Generate Data"):
    data_d = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO)
    logger.info(f"✅ Generated {len(data_d):,} records.")


# %% [markdown]
# ### 🧱 Cell 4: Model Loading OR Auto-Training
# *If settings file exists, load it. If not, auto-generate training data using heuristics.*

# %%
# [FIXED] Updated Variable Definitions for Dedupe 3.0+ API
fields = [
    dedupe.variables.String('name_only', has_missing=True),
    dedupe.variables.Categorical('gender', categories=['M', 'F', 'Other'], has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('occupation', has_missing=True),
    dedupe.variables.Exact('bank_acct_no', has_missing=True)
]

# --- Main Model Logic ---
with profile_step("Model Setup"):
    if os.path.exists(SETTINGS_FILE):
        logger.info(f"📂 Loading pre-trained settings from {SETTINGS_FILE}...")
        with open(SETTINGS_FILE, 'rb') as sf:
            deduper = dedupe.StaticDedupe(sf, num_cores=NUM_CORES)
    else:
        logger.warning(f"⚠️ Settings file not found. Starting AUTO-TRAINING (Heuristic Mode)...")
        
        # [CRITICAL FIX] We need to generate the synthetic pairs AND add them to data_d FIRST
        # AND ensure they are in the training sample.
        
        logger.info("   -> Generating synthetic training data...")
        
        match_pairs = []
        distinct_pairs = []
        training_ids = set() # Keep track of IDs involved in training
        
        # Grab a small sample of IDs for seed
        all_ids = list(data_d.keys())
        sample_ids = random.sample(all_ids, min(len(all_ids), 500))
        
        # Unique IDs for new synthetic records
        synth_counter = 900000000 
        
        # 1. Force Creation of POSITIVE Matches
        for record_id in sample_ids:
            if len(match_pairs) >= MATCH_MAX: break
            
            original_record = data_d[record_id]
            # Create a "fake" duplicate record
            fake_duplicate = original_record.copy()
            # Mutate name slightly
            fake_duplicate['name_only'] = corrupt_string(fake_duplicate['name_only'])
            
            # Register this new fake record in the main dataset!
            synth_id = str(synth_counter)
            data_d[synth_id] = fake_duplicate
            synth_counter += 1
            
            # This pair is a definite match
            match_pairs.append((original_record, fake_duplicate))
            
            # Track IDs to ensure they are in the sample
            training_ids.add(record_id)
            training_ids.add(synth_id)

        # 2. Force Creation of DISTINCT Pairs
        # Just pick two random different records
        for _ in range(DISTINCT_MAX):
            id_a, id_b = random.sample(all_ids, 2)
            if id_a != id_b:
                distinct_pairs.append((data_d[id_a], data_d[id_b]))
                training_ids.add(id_a)
                training_ids.add(id_b)

        logger.info(f"   -> Generated {len(match_pairs)} forced matches and {len(distinct_pairs)} distinct pairs.")
        logger.info(f"   -> Dataset size increased to {len(data_d):,} (added training examples).")

        # 3. NOW Initialize Dedupe
        deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
        
        # 4. Prepare Training with Custom Sample
        # We manually construct a sample dictionary that includes ALL training IDs + randoms
        logger.info("   -> Constructing custom training sample...")
        
        # Start with all records involved in training pairs
        sample_data = {uid: data_d[uid] for uid in training_ids}
        
        # Fill the rest of the sample with random records until we hit SAMPLE_SIZE
        remaining_slots = SAMPLE_SIZE - len(sample_data)
        if remaining_slots > 0:
            # Simple random sample from all_ids (might overlap, dict handles that)
            random_fill_ids = random.sample(all_ids, min(len(all_ids), remaining_slots))
            for uid in random_fill_ids:
                sample_data[uid] = data_d[uid]
        
        logger.info(f"   -> Feeding {len(sample_data)} records to prepare_training...")
        deduper.prepare_training(sample_data)
        
        # 5. Feed the synthetic data to deduper
        training_data = {'match': match_pairs, 'distinct': distinct_pairs}
        deduper.mark_pairs(training_data)
        
        logger.info("   -> Training model...")
        deduper.train()
        
        # 4. Save Settings
        with open(SETTINGS_FILE, 'wb') as sf:
            deduper.write_settings(sf)
        logger.info(f"💾 Trained model saved to {SETTINGS_FILE}")
        
        # Reload as Static for consistency
        with open(SETTINGS_FILE, 'rb') as sf:
            deduper = dedupe.StaticDedupe(sf, num_cores=NUM_CORES)


# %% [markdown]
# ### 🧱 Cell 5: Clustering
# *Runs the blocking and clustering algorithm.*

# %%
# [MODIFIED] Added monitor_cpu=True to see core usage!
with profile_step("Clustering", monitor_cpu=True):
    logger.info(f'🧩 Partitioning {len(data_d):,} records (Threshold=0.5)...')
    clustered_dupes = deduper.partition(data_d, threshold=0.5)

logger.info(f'📊 Found {len(clustered_dupes)} duplicate sets.')


# %% [markdown]
# ### 🧱 Cell 6: Write Results to CSV
# *Writes results to a CSV file instead of Database.*

# %%
with profile_step("CSV Export"):
    try:
        logger.info(f'💾 Saving results to {OUTPUT_FILE}...')
        
        # Prepare CSV Header
        csv_columns = ['cluster_id', 'cluster_score', 'cust_id', 'policy_no', 'name_only', 'address', 'dob', 'bank_acct_no', 'status']
        
        with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
            writer.writeheader()
            
            total_rows = 0
            
            for cluster_id, (cluster_members, scores) in enumerate(clustered_dupes):            
                for cust_id, score in zip(cluster_members, scores):
                    # Fetch original record details
                    rec = data_d[cust_id]
                    
                    row_out = {
                        'cluster_id': cluster_id,
                        'cluster_score': round(float(score), 4),
                        'cust_id': cust_id,
                        'policy_no': rec.get('policy_no', ''),
                        'name_only': rec.get('name_only', ''),
                        'address': rec.get('address', ''),
                        'dob': rec.get('dob', ''),
                        'bank_acct_no': rec.get('bank_acct_no', ''),
                        'status': 'Simulated'
                    }
                    
                    writer.writerow(row_out)
                    total_rows += 1
            
            logger.info(f"✅ Successfully wrote {total_rows:,} rows to {OUTPUT_FILE}")

    except Exception as e:
        logger.error(f"❌ CSV Write Error: {e}")
        raise e

logger.info('🎉 Simulation Complete.')

2025-11-27 19:26:36,983 - INFO - ✅ Environment setup complete. Asking for 18 Cores.
2025-11-27 19:26:36,984 - INFO - 💻 System Hardware: 14 Physical Cores, 20 Logical Cores available.
2025-11-27 19:26:36,985 - INFO - 📚 Fuzzy Library: rapidfuzz
2025-11-27 19:26:36,988 - INFO - ⏱️ Starting 'Generate Data'...
   [Phase 1] Generating 936,619 unique base records...
       ...10,000 records created
       ...20,000 records created
       ...30,000 records created
       ...40,000 records created
       ...50,000 records created
       ...60,000 records created
       ...70,000 records created
       ...80,000 records created
       ...90,000 records created
       ...100,000 records created
       ...110,000 records created
       ...120,000 records created
       ...130,000 records created
       ...140,000 records created
       ...150,000 records created
       ...160,000 records created
       ...170,000 records created
       ...180,000 records created
       ...190,000 records created
 